In [ ]:
import torch
import os
import torch.nn.functional as F
from collections import defaultdict
import json

In [ ]:
def token_nonsimilarity_score(
    sim: torch.Tensor,
    p: float = 3.0,
) -> torch.Tensor:

    assert sim.ndim == 3, f"Expected 3D tensor [steps, layers, tokens], got shape {tuple(sim.shape)}"

    diff = 1.0 - sim
    score = diff.pow(p).mean(dim=(0, 1)).pow(1.0 / p)

    return score


In [ ]:
def load_sim_matrix_and_transform_to_most_diff_list(folder_kv_base, filename, num_block, len_prompt, size_block):
    path_kv_file = os.path.join(folder_kv_base, filename)
    matrix_sim_step_layer_token = torch.load(path_kv_file)
    matrix_sim_step_layer_token = F.pad(matrix_sim_step_layer_token, (0,0,0,0,1,0), value=1.0)

    list_idx_sim_sorted = []

    for id_block in range(num_block):
        position_start = len_prompt + size_block * id_block
        matrix_sim_step_layer_token_cached = matrix_sim_step_layer_token[:size_block, :, :position_start]  #(steps_block, len_cached)
        score_diff_token = token_nonsimilarity_score(matrix_sim_step_layer_token_cached)
        idx_diff_token_decending = torch.argsort(score_diff_token, descending=True)
        list_idx_sim_sorted.append({'filename': filename, 'block': id_block, 'idx': idx_diff_token_decending.tolist(), 'value': score_diff_token[idx_diff_token_decending].tolist()})
    # end

    return list_idx_sim_sorted
# end

{'filename': 'batch_0_k_previous.pt',
 'block': 0,
 'idx': [88,
  8,
  101,
  120,
  63,
  60,
  111,
  90,
  50,
  80,
  96,
  104,
  97,
  69,
  46,
  30,
  74,
  93,
  37,
  43,
  22,
  89,
  65,
  76,
  52,
  127,
  55,
  73,
  71,
  72,
  7,
  11,
  70,
  41,
  44,
  45,
  126,
  77,
  61,
  51,
  53,
  64,
  85,
  57,
  17,
  121,
  56,
  84,
  116,
  38,
  47,
  123,
  68,
  23,
  109,
  94,
  0,
  59,
  9,
  5,
  15,
  118,
  49,
  54,
  110,
  108,
  112,
  48,
  119,
  107,
  14,
  27,
  81,
  106,
  102,
  3,
  19,
  78,
  122,
  79,
  13,
  36,
  75,
  58,
  62,
  113,
  103,
  6,
  2,
  28,
  42,
  115,
  82,
  105,
  98,
  40,
  35,
  31,
  86,
  124,
  67,
  21,
  91,
  1,
  114,
  87,
  99,
  24,
  117,
  83,
  33,
  26,
  125,
  34,
  25,
  29,
  4,
  66,
  16,
  100,
  39,
  20,
  32,
  92,
  12,
  10,
  18,
  95],
 'value': [0.13507293164730072,
  0.08493713289499283,
  0.06994315981864929,
  0.06480918079614639,
  0.05888314172625542,
  0.056583695113658905,
  0.053

In [ ]:
folder_kv_base = 'sims_kv_test'
filename_report = 'all_in_one_sim_report.json'

len_prompt = 128
num_block = 4
len_target = 256
size_block = int(len_target / num_block)

dict_filename_to_list_idx_sorted = defaultdict(list)

for filename in os.listdir(folder_kv_base):
    if filename[0] == '.':
        continue
    # end

    # matrix_sim_step_layer_token, num_block, len_prompt, size_block, path_kv_base, filename
    list_diff_sorted = load_sim_matrix_and_transform_to_most_diff_list(folder_kv_base, filename, num_block, len_prompt, size_block)
    dict_filename_to_list_idx_sorted[filename] = list_diff_sorted
# end

with open(filename_report, 'w+') as file:
    file.write(json.dumps(dict_filename_to_list_idx_sorted))
# end

